# SHAP explanations for the primary full-M0 models and paired weather ablation

Main explanations use the exact Notebook 06 primary full-M0 models: P95 XGBoost and P90 Random Forest. Notebook 08 paired M0/M1 explanations are secondary for RQ3. Local TP/TN/FP/FN cases use primary P95 and its Notebook 08B test predictions. No model is refit and no threshold changes.

Tree SHAP explains the base estimator output. Platt calibration subsequently transforms that output into the reported risk probability; SHAP does not decompose calibrated probability. Attributions describe fitted model behavior and are not causal effects.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
from sklearn.pipeline import Pipeline
from IPython.display import display

ROOT=Path.cwd().resolve()
while not (ROOT/"data"/"processed").exists() and ROOT!=ROOT.parent: ROOT=ROOT.parent
if not (ROOT/"data"/"processed").exists(): raise RuntimeError("Could not locate project root")
DATA_DIR=ROOT/"data"/"processed"
MODEL_DIR=ROOT/"models"/"08_weather_model_comparison"
PRIMARY_MODEL_DIR=ROOT/"models"/"06_calibrated_m0"
PRIMARY_LOCK_PATH=ROOT/"reports"/"tables"/"06_calibration_evaluation"/"06_calibration_and_threshold_manifest.json"
TABLE_DIR=ROOT/"reports"/"tables"/"09_shap_analysis"
FIGURE_DIR=ROOT/"reports"/"figures"/"09_shap_analysis"
for d in (TABLE_DIR,FIGURE_DIR): d.mkdir(parents=True,exist_ok=True)
BUNDLE_PATHS={"PRIMARY_P95_M0":PRIMARY_MODEL_DIR/"p95_calibrated_m0.joblib","PRIMARY_P90_M0":PRIMARY_MODEL_DIR/"p90_calibrated_m0.joblib",
 "PAIRED_P95_M0":MODEL_DIR/"p95_m0_paired_weather.joblib","PAIRED_P95_M1":MODEL_DIR/"p95_m1_paired_weather.joblib",
 "PAIRED_P90_M0":MODEL_DIR/"p90_m0_paired_weather.joblib","PAIRED_P90_M1":MODEL_DIR/"p90_m1_paired_weather.joblib"}
DATA_PATHS={"PRIMARY_P95_M0":DATA_DIR/"05_model_ready_m0_p95.csv","PRIMARY_P90_M0":DATA_DIR/"05_model_ready_m0_p90.csv",
 "PAIRED_P95_M0":DATA_DIR/"05_model_ready_m0_p95.csv","PAIRED_P95_M1":DATA_DIR/"07_model_ready_m1_p95.csv",
 "PAIRED_P90_M0":DATA_DIR/"05_model_ready_m0_p90.csv","PAIRED_P90_M1":DATA_DIR/"07_model_ready_m1_p90.csv"}
PREDICTIONS_PATH=ROOT/"reports"/"tables"/"08b_final_test_evaluation"/"08b_final_test_row_predictions.csv"
FINAL_MANIFEST_PATH=ROOT/"reports"/"tables"/"08b_final_test_evaluation"/"08b_final_test_evaluation_manifest.json"
LOCK_MANIFEST_PATH=ROOT/"reports"/"tables"/"08_weather_model_comparison"/"08_paired_weather_experiment_manifest.json"
PRIMARY_LOCK_MANIFEST_PATH=PRIMARY_LOCK_PATH
RANDOM_STATE=42
MAX_EXPLANATION_ROWS=500
TOP_FEATURES=15

def read_json(path): return json.loads(Path(path).read_text(encoding="utf-8"))
def numeric_x(frame,features): return frame[features].apply(lambda c:pd.to_numeric(c,errors="coerce").astype("float64"))
def probability_logit(prob):
 p=np.clip(np.asarray(prob,dtype=float),1e-6,1-1e-6)
 return np.log(p/(1-p)).reshape(-1,1)

print("SHAP version:",shap.__version__,"| deterministic global sample limit:",MAX_EXPLANATION_ROWS)


## 1. Verify the final-evaluation handoff and choose the shared sample

The final prediction handoff includes primary full-M0 and paired-weather rows. A deterministic sample of up to 500 test_seen IDs is selected from the intersection so the primary explanation and secondary weather-ablation explanations can be compared on the same observations. TP/TN/FP/FN cases are selected from the primary full-M0 P95 predictions.

In [ ]:
for p in [*BUNDLE_PATHS.values(),*DATA_PATHS.values(),PREDICTIONS_PATH,FINAL_MANIFEST_PATH,LOCK_MANIFEST_PATH,PRIMARY_LOCK_MANIFEST_PATH]:
 if not p.exists(): raise FileNotFoundError(f"Required handoff missing: {p}")
final_manifest=read_json(FINAL_MANIFEST_PATH); lock=read_json(LOCK_MANIFEST_PATH); primary_lock=read_json(PRIMARY_LOCK_MANIFEST_PATH)
if not final_manifest.get("validation_passed",False) or not lock.get("validation_passed",False) or not primary_lock.get("validation_passed",False): raise ValueError("Final evaluation or model lock validation failed")
predictions=pd.read_csv(PREDICTIONS_PATH,dtype={"row_id":"string","household_id":"string"},parse_dates=["measurement_date"])
if predictions.duplicated(["target_name","dataset","row_id"]).any(): raise ValueError("Duplicate row-level predictions")
if not {"M0","M1","PRIMARY_FULL_M0"}.issubset(set(predictions.dataset.unique())) or not set(predictions.target_name.unique())=={"P95","P90"}: raise ValueError("Expected primary full-M0 and paired P95/P90 predictions")

p95_m0=predictions.loc[predictions.target_name.eq("P95")&predictions.dataset.eq("PRIMARY_FULL_M0")&predictions.test_cohort.eq("test_seen")].copy()
if p95_m0.empty: raise ValueError("No primary P95 full-M0 test_seen cases in the final-evaluation handoff")
case_specs={
 "true_positive":p95_m0.loc[p95_m0.predicted_label.eq(1)&p95_m0.target.eq(1)].sort_values("calibrated_probability",ascending=False),
 "true_negative":p95_m0.loc[p95_m0.predicted_label.eq(0)&p95_m0.target.eq(0)].sort_values("calibrated_probability"),
 "false_positive":p95_m0.loc[p95_m0.predicted_label.eq(1)&p95_m0.target.eq(0)].assign(_margin=lambda d:(d.calibrated_probability-d.locked_threshold).abs()).sort_values("_margin"),
 "false_negative":p95_m0.loc[p95_m0.predicted_label.eq(0)&p95_m0.target.eq(1)].assign(_margin=lambda d:(d.calibrated_probability-d.locked_threshold).abs()).sort_values("_margin"),
}
if any(part.empty for part in case_specs.values()): raise ValueError("primary P95 full-M0 test_seen does not contain all TP/TN/FP/FN cases")
local_cases=pd.concat([part.head(1).assign(local_case=case) for case,part in case_specs.items()],ignore_index=True)
local_cases["model_role"]="primary_full_m0"; local_cases["research_questions"]="RQ1,RQ2,RQ4"
local_cases=local_cases[["local_case","model_role","research_questions","row_id","household_id","measurement_date","target","raw_base_probability","calibrated_probability","locked_threshold","predicted_label"]]
local_cases.to_csv(TABLE_DIR/"09_p95_m0_local_test_cases.csv",index=False)

all_seen_ids=set(p95_m0.row_id.astype(str))
paired_seen_ids=set(predictions.loc[predictions.target_name.eq("P95")&predictions.dataset.eq("M0")&predictions.test_cohort.eq("test_seen"),"row_id"].astype(str))
common_seen_ids=sorted(set(p95_m0.row_id.astype(str))&paired_seen_ids)
if not common_seen_ids: raise ValueError("No shared primary/paired test_seen rows for SHAP")
sample_ids=sorted(pd.Series(common_seen_ids).sample(n=min(MAX_EXPLANATION_ROWS,len(common_seen_ids)),random_state=RANDOM_STATE).tolist())
explanation_ids=sorted(set(sample_ids)|set(local_cases.row_id.astype(str)))
pd.DataFrame({"row_id":sample_ids}).to_csv(TABLE_DIR/"09_shap_sample_row_ids.csv",index=False)
display(local_cases); print("Eligible test_seen rows:",len(all_seen_ids),"| common global explanation sample:",len(sample_ids),"| local cases:",len(local_cases))

bundles={key:joblib.load(path) for key,path in BUNDLE_PATHS.items()}
for key,bundle in bundles.items():
 if key.startswith("PRIMARY_"):
  target=key.split("_")[1]
  if bundle.get("model_role")!="primary_full_m0" or bundle["threshold"]!=primary_lock["selected_models"][target]["threshold"]: raise ValueError(f"Invalid primary bundle {key}")
 else:
  paired_key=key.replace("PAIRED_","")
  if bundle.get("model_role")!="controlled_weather_ablation" or bundle.get("outer_test_used") is not False: raise ValueError(f"Invalid paired bundle {key}")
  if bundle["threshold"]!=lock["models"][paired_key]["threshold"]: raise ValueError(f"{key} threshold differs from paired model lock")


## 2. Global SHAP explanations

The same shared sample is explained for both primary Notebook 06 full-M0 models and all four secondary Notebook 08 paired-weather models. The saved preprocessing pipeline transforms the raw features exactly as during scoring; TreeExplainer explains the underlying tree estimator. Mean absolute SHAP is a ranking of model attribution magnitude, not an effect size.

In [ ]:
def read_selected_rows(path,ids,features):
 ids=set(map(str,ids)); rowids=pd.read_csv(path,usecols=["row_id"],dtype={"row_id":"string"}).row_id
 positions=np.flatnonzero(rowids.isin(ids).to_numpy()); found=set(rowids.iloc[positions].astype(str))
 if found!=ids: raise ValueError(f"{path.name}: expected explanation row IDs missing={len(ids-found)}")
 keep=set(positions.tolist()); cols=list(dict.fromkeys(["row_id","household_id","measurement_date","target",*features]))
 out=pd.read_csv(path,usecols=cols,dtype={"row_id":"string","household_id":"string"},skiprows=lambda line:line>0 and (line-1) not in keep)
 if set(out.row_id.astype(str))!=ids or out.row_id.duplicated().any(): raise ValueError(f"{path.name}: selective explanation rows invalid")
 out["measurement_date"]=pd.to_datetime(out.measurement_date,errors="raise")
 return out.set_index("row_id").loc[sorted(ids)].reset_index()

def normalize_values(explainer,x):
 raw=explainer.shap_values(x)
 if isinstance(raw,list): raw=raw[1] if len(raw)>1 else raw[0]
 values=np.asarray(raw)
 if values.ndim==3: values=values[:,:,1] if values.shape[-1]>1 else values[:,:,0]
 expected=np.asarray(explainer.expected_value)
 base=float(expected.reshape(-1)[1] if expected.size>1 else expected.reshape(-1)[0])
 if values.shape!=x.shape: raise ValueError(f"SHAP dimensions {values.shape} differ from transformed feature matrix {x.shape}")
 if not np.isfinite(values).all(): raise ValueError("Non-finite SHAP values")
 return values,base

shap_values_by_model={}; transformed_by_model={}; feature_names_by_model={}; raw_frames={}; explainers={}; importance_rows=[]; plot_paths=[]
for key,bundle in bundles.items():
 frame=read_selected_rows(DATA_PATHS[key],sample_ids,bundle["features"])
 if not set(frame.row_id.astype(str)).issubset(all_seen_ids): raise ValueError(f"{key}: global explanations include a non-test_seen row")
 raw_frames[key]=frame
 pipeline=bundle["base_estimator"]
 if not isinstance(pipeline,Pipeline) or "imputer" not in pipeline.named_steps or "model" not in pipeline.named_steps: raise TypeError(f"{key}: expected the saved imputer/tree pipeline")
 x_raw=numeric_x(frame,bundle["features"]); imputer=pipeline.named_steps["imputer"]
 x=imputer.transform(x_raw); feature_names=np.asarray(imputer.get_feature_names_out(bundle["features"]),dtype=str)
 explainer=shap.TreeExplainer(pipeline.named_steps["model"],feature_perturbation="tree_path_dependent")
 values,base=normalize_values(explainer,x)
 shap_values_by_model[key]=values; transformed_by_model[key]=x; feature_names_by_model[key]=feature_names; explainers[key]=explainer
 mean_abs=np.abs(values).mean(axis=0); mean_signed=values.mean(axis=0)
 for name,ma,ms in zip(feature_names,mean_abs,mean_signed): importance_rows.append({"model":key,"target":bundle["target"],"dataset":bundle.get("dataset","PRIMARY_FULL_M0"),"feature":str(name),"mean_abs_shap_raw_output":float(ma),"mean_signed_shap_raw_output":float(ms),"explained_rows":len(frame)})

 exp=shap.Explanation(values=values,base_values=np.repeat(base,len(frame)),data=x,feature_names=feature_names)
 plt.figure(figsize=(10,7)); shap.plots.beeswarm(exp,max_display=20,show=False); plt.title(f"{key}: SHAP beeswarm")
 plt.tight_layout(); fname=f"{key.lower()}_shap_beeswarm.png"; plt.savefig(FIGURE_DIR/fname,dpi=160,bbox_inches="tight"); plt.close(); plot_paths.append(fname)
 top=np.argsort(mean_abs)[-TOP_FEATURES:]; order=top[np.argsort(mean_abs[top])]
 fig,ax=plt.subplots(figsize=(9,6)); ax.barh(feature_names[order],mean_abs[order],color="#376a9e")
 ax.set(title=f"{key}: top {TOP_FEATURES} mean absolute SHAP",xlabel="Mean absolute SHAP (raw model output)"); fig.tight_layout()
 fname=f"{key.lower()}_top15_shap_importance.png"; fig.savefig(FIGURE_DIR/fname,dpi=160,bbox_inches="tight"); plt.close(fig); plot_paths.append(fname)
 print(key,"features:",len(feature_names),"rows:",len(frame))

importance=pd.DataFrame(importance_rows)
importance["rank_within_model"]=importance.groupby("model")["mean_abs_shap_raw_output"].rank(method="first",ascending=False).astype(int)
importance=importance.sort_values(["model","rank_within_model"])
importance.to_csv(TABLE_DIR/"09_mean_absolute_shap_importance.csv",index=False)
display(importance.groupby("model",as_index=False).head(TOP_FEATURES))


## 3. Primary full-M0 P95 dependence plots and secondary weather attribution

Dependence plots use the highest-ranked historical predictors from the newly computed primary P95 full-M0 SHAP ranking. The paired-weather M1 attribution table is secondary: a high SHAP contribution means the model used the feature in this sample, not that weather caused the outcome or improved test performance.

In [ ]:
p95_key="PRIMARY_P95_M0"; p95_names=feature_names_by_model[p95_key]; p95_values=shap_values_by_model[p95_key]; p95_x=transformed_by_model[p95_key]
historical_names=[name for name in p95_names if any(token in name for token in ("peak","p95_high","days_since_last_high"))]
primary_ranked=importance.loc[importance.model.eq("PRIMARY_P95_M0")&importance.feature.isin(historical_names)].sort_values("mean_abs_shap_raw_output",ascending=False)
requested=primary_ranked.feature.head(4).tolist()
available=[name for name in requested if name in set(p95_names)]
if not available: raise ValueError("None of the planned P95 historical dependence features were found")
for feature in available:
 index=int(np.flatnonzero(p95_names==feature)[0])
 plt.figure(figsize=(8,5)); shap.dependence_plot(index,p95_values,p95_x,feature_names=p95_names,interaction_index="auto",show=False)
 plt.title(f"P95 M0 SHAP dependence: {feature}"); plt.tight_layout()
 fname=f"p95_m0_dependence_{feature}.png"; plt.savefig(FIGURE_DIR/fname,dpi=160,bbox_inches="tight"); plt.close(); plot_paths.append(fname)

weather_features=sorted(set(bundles["PAIRED_P95_M1"]["features"])-set(bundles["PAIRED_P95_M0"]["features"]))
weather_summary=importance.loc[importance.model.isin(["PAIRED_P95_M1","PAIRED_P90_M1"])&importance.feature.isin(weather_features)].copy()
weather_summary=weather_summary.sort_values(["model","mean_abs_shap_raw_output"],ascending=[True,False])
weather_summary.to_csv(TABLE_DIR/"09_weather_feature_shap_attribution.csv",index=False)
display(weather_summary)


## 4. Local primary P95 full-M0 explanations

One true positive and true negative, plus false-positive and false-negative cases nearest the locked threshold, are explained with waterfall plots from the primary full-M0 P95 estimator. The companion table reports actual outcome, predicted class, raw base-model probability, calibrated probability, and the frozen threshold. The waterfall contributions remain on the underlying estimator output scale.

In [ ]:
local_rows=[]
local_plot_paths=[]
local_feature_names=feature_names_by_model["PRIMARY_P95_M0"]
local_imputer=bundles["PRIMARY_P95_M0"]["base_estimator"].named_steps["imputer"]
local_estimator=bundles["PRIMARY_P95_M0"]["base_estimator"].named_steps["model"]
local_explainer=shap.TreeExplainer(local_estimator,feature_perturbation="tree_path_dependent")
for _,case in local_cases.iterrows():
 rid=str(case.row_id); frame=read_selected_rows(DATA_PATHS["PRIMARY_P95_M0"],[rid],bundles["PRIMARY_P95_M0"]["features"])
 x=local_imputer.transform(numeric_x(frame,bundles["PRIMARY_P95_M0"]["features"]))
 vals,base=normalize_values(local_explainer,x); vals=vals[0]
 local_value_map=dict(zip(local_feature_names,x[0]))
 order_pos=np.argsort(vals)[::-1]; order_neg=np.argsort(vals)
 for sign,indices in (("positive",[i for i in order_pos if vals[i]>0][:5]),("negative",[i for i in order_neg if vals[i]<0][:5])):
  for rank,index in enumerate(indices,1):
   local_rows.append({"local_case":case.local_case,"model_role":"primary_full_m0","research_questions":"RQ1,RQ2,RQ4","row_id":rid,"household_id":case.household_id,"target":int(case.target),
                      "predicted_label":int(case.predicted_label),"raw_base_probability":float(case.raw_base_probability),"calibrated_risk":float(case.calibrated_probability),"threshold":float(case.locked_threshold),
                      "shap_direction":sign,"rank_within_direction":rank,"feature":str(local_feature_names[index]),
                      "shap_value_raw_output":float(vals[index]),"transformed_feature_value":float(local_value_map[local_feature_names[index]])})
 exp=shap.Explanation(values=vals,base_values=base,data=x[0],feature_names=local_feature_names)
 plt.figure(figsize=(9,6)); shap.plots.waterfall(exp,max_display=15,show=False)
 plt.title(f"Primary P95 full M0 {case.local_case}: calibrated risk={case.calibrated_probability:.3f}; threshold={case.locked_threshold:.2f}; actual={int(case.target)}")
 plt.tight_layout(); fname=f"primary_p95_full_m0_local_{case.local_case}.png"; plt.savefig(FIGURE_DIR/fname,dpi=160,bbox_inches="tight"); plt.close(); local_plot_paths.append(fname)

local_attribution=pd.DataFrame(local_rows)
local_attribution.to_csv(TABLE_DIR/"09_p95_m0_local_shap_attributions.csv",index=False)
display(local_attribution)


## 5. Validate SHAP artifacts and record limitations

In [ ]:
checks=[]
def add_check(name,passed,detail): checks.append({"check":name,"passed":bool(passed),"detail":str(detail)})
add_check("final_evaluation_and_both_model_locks_passed",final_manifest.get("validation_passed",False) and lock.get("validation_passed",False) and primary_lock.get("validation_passed",False),"Loaded validated primary and paired model locks")
add_check("shared_test_seen_sample_used",len(sample_ids)==min(MAX_EXPLANATION_ROWS,len(p95_m0)) and all(set(raw_frames[k].row_id.astype(str))==set(sample_ids) for k in bundles),len(sample_ids))
add_check("all_six_model_roles_explained",set(shap_values_by_model)==set(BUNDLE_PATHS),sorted(shap_values_by_model))
add_check("shap_values_finite_and_shape_matched",all(np.isfinite(shap_values_by_model[k]).all() and shap_values_by_model[k].shape[1]==len(feature_names_by_model[k]) for k in bundles),{k:shap_values_by_model[k].shape for k in bundles})
add_check("four_primary_local_test_error_types_explained",set(local_cases.local_case)=={"true_positive","true_negative","false_positive","false_negative"} and len(local_plot_paths)==4,local_plot_paths)
add_check("weather_features_reported_separately",set(weather_summary.model.unique())=={"PAIRED_P95_M1","PAIRED_P90_M1"},weather_features)
add_check("model_roles_and_thresholds_match_locks",all(bundles[k]["threshold"]==(primary_lock["selected_models"][k.split("_")[1]]["threshold"] if k.startswith("PRIMARY_") else lock["models"][k.replace("PAIRED_","")]["threshold"]) for k in bundles),"Exact locked primary and paired bundles reused")
validation=pd.DataFrame(checks); display(validation); validation.to_csv(TABLE_DIR/"09_final_validation.csv",index=False)
if not validation.passed.all(): raise AssertionError("Notebook 09 validation failed")

def sha256(path):
 h=hashlib.sha256()
 with Path(path).open("rb") as f:
  for block in iter(lambda:f.read(8*1024*1024),b""): h.update(block)
 return h.hexdigest()
manifest={"created_utc":datetime.now(timezone.utc).isoformat(),"stage":"09_final_shap_analysis",
 "primary_models":["PRIMARY_P95_M0","PRIMARY_P90_M0"],"primary_role":"primary_full_m0","secondary_models":["PAIRED_P95_M0","PAIRED_P95_M1","PAIRED_P90_M0","PAIRED_P90_M1"],"secondary_role":"controlled_weather_ablation",
 "explanation_sample":{"cohort":"test_seen","rows":len(sample_ids),"eligible_rows":len(all_seen_ids),"random_state":RANDOM_STATE},
 "local_cases":[{"case":str(r.local_case),"model_role":"primary_full_m0","research_questions":"RQ1,RQ2,RQ4","row_id":str(r.row_id),"raw_base_probability":float(r.raw_base_probability),"calibrated_risk":float(r.calibrated_probability),"threshold":float(r.locked_threshold),"actual":int(r.target),"prediction":int(r.predicted_label)} for _,r in local_cases.iterrows()],
 "explanation_output":"Tree SHAP raw model output; Platt calibrator used only for displayed risks",
 "bundle_sha256":{key:sha256(path) for key,path in BUNDLE_PATHS.items()},"final_test_predictions_sha256":sha256(PREDICTIONS_PATH),
 "summary_plots":plot_paths,"local_waterfalls":local_plot_paths,"validation_passed":bool(validation.passed.all()),"checks":checks,
 "limitations":["SHAP describes fitted model behavior and is not causal evidence.","Global explanations use a deterministic sample of up to 500 test_seen rows.","The natural unseen-household cohort is not used for global explanations.","M1 weather attribution is secondary and does not establish weather benefit."]}
(TABLE_DIR/"09_final_shap_manifest.json").write_text(json.dumps(manifest,indent=2),encoding="utf-8")
print("SHAP interpretation complete. Primary explanation: P95 M0; M1 and P90 are secondary/sensitivity.")


## Handoff

The primary explanations are for the exact Notebook 06 P95/P90 full-M0 models evaluated in Notebook 08B. The Notebook 08 paired-weather P95/P90 M0/M1 explanations are secondary and support RQ3. Use Notebook 08B for performance claims; SHAP describes model behavior, not causal effects.